### Fixing SSL Certificate Verification Using `truststore`

```python
!pip install truststore

import truststore
truststore.inject_into_ssl()
```

**1. `!pip install truststore`**
- Installs the `truststore` Python package.
- It allows Python to use the operating system's native SSL/TLS certificate store.

**2. `import truststore`**
- Imports the installed `truststore` package into the Python environment.

**3. `truststore.inject_into_ssl()`**
- Integrates the operating system's certificate verification system into Python's default SSL handling.
- On Windows, this allows Python to use certificates trusted by the Windows certificate store.
- It can resolve SSL certificate verification errors when connecting securely to services such as Neo4j AuraDB.

**Why is it needed?**

In my GraphRAG project, connecting to Neo4j AuraDB using `neo4j+s://` resulted in an SSL certificate verification error. Using `truststore` allowed Python to use the system's trusted certificates instead of relying solely on its default certificate configuration.

**Important:** Run `truststore.inject_into_ssl()` early in the notebook, before initializing the Neo4j connection. Restart the kernel after installing the package if necessary.

In [ ]:
# ! pip install -U langchain-huggingface sentence-transformers

   ---------------------------------------- 0.0/573.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/573.8 kB ? eta -:--:--
   ------------------ --------------------- 262.1/573.8 kB ? eta -:--:--
   ---------------------------------------- 573.8/573.8 kB 938.2 kB/s  0:00:00
   ---------------------------------------- 0.0/869.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/869.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/869.0 kB ? eta -:--:--
   ------------ --------------------------- 262.1/869.0 kB ? eta -:--:--
   ------------ --------------------------- 262.1/869.0 kB ? eta -:--:--
   ---------------------- --------------- 524.3/869.0 kB 494.0 kB/s eta 0:00:01
   ---------------------- --------------- 524.3/869.0 kB 494.0 kB/s eta 0:00:01
   ---------------------------------- --- 786.4/869.0 kB 523.2 kB/s eta 0:00:01
   ---------------------------------------- 869.0/869.0 kB 500.8 kB/s  0:00:01

  Attempting uninstall: lan

In [ ]:
# ! pip install truststore

In [1]:
import truststore
truststore.inject_into_ssl()

In [14]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph, Neo4jVector

In [15]:
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY is missing")

if not NEO4J_URI:
    raise ValueError("NEO4J_URI is missing")

if not NEO4J_USERNAME:
    raise ValueError("NEO4J_USERNAME is missing")

if not NEO4J_PASSWORD:
    raise ValueError("NEO4J_PASSWORD is missing")

if not NEO4J_DATABASE:
    raise ValueError("NEO4J_DATABASE is missing")

In [16]:
# temperature=0 ensures deterministic entity and relationship extraction
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
# PyPDFLoader yields one Document per page
loader = PyPDFLoader("../data/elon_musk.pdf")
pages = loader.load()

for i, p in enumerate(pages):
    print(f"Page {i + 1}: {len(p.page_content)} chars")

incorrect startxref pointer(1)
parsing for Object Streams


Page 1: 2343 chars
Page 2: 1192 chars


In [6]:
# smaller chunks give the LLM tighter context for entity extraction
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"{len(chunks)} chunks created")

14 chunks created


In [7]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    refresh_schema=False,
)

In [8]:
graph_transformer = LLMGraphTransformer(llm=llm)

In [9]:
graph_docs = graph_transformer.convert_to_graph_documents(chunks)

print(f"{len(graph_docs)} graph documents extracted")

# spot-check the first extraction
print("Nodes:", [n.id for n in graph_docs[0].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[0].relationships])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


14 graph documents extracted
Nodes: ['Elon Musk', 'Pretoria', 'South Africa']
Rels:  [('Elon Musk', 'BORN_IN', 'Pretoria'), ('Pretoria', 'LOCATED_IN', 'South Africa')]


In [11]:
# include_source=True links each entity node back to its source Document node,
# which is required for Neo4jVector.from_existing_graph in the next cell
graph.add_graph_documents(
    graph_docs,
    include_source=True,
    baseEntityLabel=True
)
print("Graph stored in Neo4J")

Graph stored in Neo4J


In [18]:
# create a vector index over the Document nodes stored above
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    index_name="elon_musk_chunks",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
print("Vector index created")

Vector index created


In [19]:
# verify what landed in Neo4J
node_counts = graph.query(
    "MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC"
)
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC"
)
print("Nodes:")
for r in node_counts:
    print(" ", r)
print("Relationships:")
for r in rel_counts:
    print(" ", r)

[#D37A]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-819cf3afaca3-1-0185.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)
Transaction failed and will be retried in 1.085340626293759s (Failed to read from defunct connection IPv4Address(('p-mt-819cf3afaca3-1-0185.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))


Nodes:
  {'label': ['__Entity__', 'Person'], 'count': 24}
  {'label': ['__Entity__', 'Location'], 'count': 16}
  {'label': ['Document'], 'count': 14}
  {'label': ['__Entity__', 'Organization'], 'count': 12}
  {'label': ['__Entity__', 'Product'], 'count': 10}
  {'label': ['__Entity__', 'Service'], 'count': 1}
  {'label': ['__Entity__', 'Concept'], 'count': 1}
Relationships:
  {'type': 'MENTIONS', 'count': 102}
  {'type': 'PARENT', 'count': 26}
  {'type': 'LOCATED_IN', 'count': 12}
  {'type': 'FOUNDER', 'count': 11}
  {'type': 'HEADQUARTERED', 'count': 7}
  {'type': 'MANUFACTURES', 'count': 7}
  {'type': 'CEO', 'count': 4}
  {'type': 'RESIDENT', 'count': 4}
  {'type': 'PART_OF', 'count': 3}
  {'type': 'ACQUIRED', 'count': 3}
  {'type': 'HEADQUARTERED_IN', 'count': 3}
  {'type': 'PARTNER', 'count': 2}
  {'type': 'SIBLING', 'count': 2}
  {'type': 'BORN_IN', 'count': 1}
  {'type': 'SPOUSE', 'count': 1}
  {'type': 'CHIEF_ENGINEER', 'count': 1}
  {'type': 'CO_FOUNDER', 'count': 1}
  {'type': 

In [21]:
graph.query("""
    MATCH (d:Document)
    RETURN keys(d) AS properties
    LIMIT 5
""")

[#D437]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-819cf3afaca3-1-0185.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)
Transaction failed and will be retried in 1.0898165525393964s (Failed to read from defunct connection IPv4Address(('p-mt-819cf3afaca3-1-0185.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))


[{'properties': ['id',
   'text',
   'creator',
   'creationdate',
   'keywords',
   'trapped',
   'author',
   'subject',
   'source',
   'total_pages',
   'title',
   'moddate',
   'producer',
   'page',
   'page_label',
   'embedding']},
 {'properties': ['id',
   'text',
   'creator',
   'creationdate',
   'keywords',
   'trapped',
   'author',
   'subject',
   'source',
   'total_pages',
   'title',
   'moddate',
   'producer',
   'page',
   'page_label',
   'embedding']},
 {'properties': ['id',
   'text',
   'creator',
   'creationdate',
   'keywords',
   'trapped',
   'author',
   'subject',
   'source',
   'total_pages',
   'title',
   'moddate',
   'producer',
   'page',
   'page_label',
   'embedding']},
 {'properties': ['id',
   'text',
   'creator',
   'creationdate',
   'keywords',
   'trapped',
   'author',
   'subject',
   'source',
   'total_pages',
   'title',
   'moddate',
   'producer',
   'page',
   'page_label',
   'embedding']},
 {'properties': ['id',
   'text',
  

In [20]:
metadata = graph.query("""
    MATCH (d:Document)
    RETURN d.id AS id,
           d.source AS source,
           d.page AS page,
           d.text AS text,
           d.embedding IS NOT NULL AS has_embedding
    LIMIT 5
""")

for row in metadata:
    print(row)

[#D3EC]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-819cf3afaca3-1-0185.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)
Transaction failed and will be retried in 0.9360091127573047s (Failed to read from defunct connection IPv4Address(('p-mt-819cf3afaca3-1-0185.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))


{'id': '48131a6fe23993823a6c6ea4785e962d', 'source': '../data/elon_musk.pdf', 'page': 0, 'text': "Elon Musk: A Profile\n1. Overview\nElon Musk is an American entrepreneur and engineer born on June 28, 1971, in Pretoria, South\nAfrica. Elon Musk holds American nationality. Elon Musk has been recognised as the world's\nwealthiest person on multiple occasions.\n2. Family Background", 'has_embedding': True}
{'id': '12ac61bd45c1c128434d9df4fd807cb5', 'source': '../data/elon_musk.pdf', 'page': 0, 'text': "2. Family Background\nElon Musk's father is Errol Musk, a South African engineer. Elon Musk's mother is Maye Musk, a\nCanadian-South African model and dietitian. Elon Musk's brother is Kimbal Musk, an entrepreneur.\nElon Musk's sister is Tosca Musk, a filmmaker based in Los Angeles, United States.", 'has_embedding': True}
{'id': '514440b17b629ec4e2131183bc62b0fd', 'source': '../data/elon_musk.pdf', 'page': 0, 'text': '3. Personal Life\nElon Musk was married to Justine Wilson from 2000 to 20

In [22]:
metadata = graph.query("""
    MATCH (d:Document)
    RETURN
        d.id AS id,
        d.source AS source,
        d.page AS page,
        d.page_label AS page_label,
        d.total_pages AS total_pages,
        left(d.text, 100) AS text_preview
    LIMIT 5
""")

for row in metadata:
    print(row)

{'id': '48131a6fe23993823a6c6ea4785e962d', 'source': '../data/elon_musk.pdf', 'page': 0, 'page_label': '1', 'total_pages': 2, 'text_preview': 'Elon Musk: A Profile\n1. Overview\nElon Musk is an American entrepreneur and engineer born on June 28,'}
{'id': '12ac61bd45c1c128434d9df4fd807cb5', 'source': '../data/elon_musk.pdf', 'page': 0, 'page_label': '1', 'total_pages': 2, 'text_preview': "2. Family Background\nElon Musk's father is Errol Musk, a South African engineer. Elon Musk's mother "}
{'id': '514440b17b629ec4e2131183bc62b0fd', 'source': '../data/elon_musk.pdf', 'page': 0, 'page_label': '1', 'total_pages': 2, 'text_preview': '3. Personal Life\nElon Musk was married to Justine Wilson from 2000 to 2008. Elon Musk and Justine Wi'}
{'id': '301387de8c601ea6fd6313e6474dd767', 'source': '../data/elon_musk.pdf', 'page': 0, 'page_label': '1', 'total_pages': 2, 'text_preview': "Elon Musk's partner is Grimes, whose full name is Claire Boucher. Elon Musk and Grimes have\nthree ch"}
{'id': '4de